In [ ]:
# VENDOR PERFORMANCE ANALYTICS & TIER ASSIGNMENT SYSTEM
# Complete Analytics Pipeline with CSV Integration




# Install required packages
!pip install pandas numpy scipy scikit-learn seaborn matplotlib plotly xgboost shap missingno -q

# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy import stats
import warnings
import io
from google.colab import files
import missingno as msno

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# 2. DATA LOADING FROM CSV/UPLOAD


def load_vendor_data():
    """Load vendor master data from CSV or create from sample"""
    try:
        #  uploading the  CSV
        print("📤 Please upload your Vendor Master CSV file:")
        uploaded = files.upload()

        for filename in uploaded.keys():
            if filename.endswith('.csv'):
                vendor_df = pd.read_csv(io.BytesIO(uploaded[filename]))
                print(f"✅ Vendor data loaded from: {filename}")
                return vendor_df
    except:
        print("⚠️ No vendor CSV uploaded. Using sample data...")
        # Create sample vendor data
        vendor_data = {
            'supplier_code': ['H001', 'H002', 'H003', 'H004', 'H005', 'H006', 'H007', 'H008', 'H009', 'H010'],
            'vendor_name': ['Alpha Materials', 'Beta Chemicals', 'Gamma Logistics', 'Delta Packaging',
                           'Epsilon Supply', 'Zeta Raw Materials', 'Eta Chemicals', 'Theta Logistics',
                           'Iota Packaging', 'Kappa Supply'],
            'vendor_category': ['Raw Material', 'Chemical', 'Logistics', 'Packaging', 'Raw Material',
                               'Chemical', 'Logistics', 'Packaging', 'Raw Material', 'Chemical'],
            'vendor_location': ['Mumbai', 'Delhi', 'Chennai', 'Bangalore', 'Hyderabad',
                               'Pune', 'Mumbai', 'Delhi', 'Chennai', 'Bangalore'],
            'payment_terms': [30, 60, 45, 30, 60, 45, 30, 60, 45, 30],
            'vendor_since': ['2020-01-15', '2019-05-20', '2021-03-10', '2018-11-05',
                            '2022-02-28', '2019-08-15', '2020-09-22', '2021-12-10',
                            '2018-06-30', '2023-01-05']
        }
        vendor_df = pd.DataFrame(vendor_data)

    return vendor_df

def load_po_data():
    """Load purchase order data from CSV or create from sample"""
    try:
        #upload CSV
        print("\n📤 Please upload your Purchase Orders CSV file:")
        uploaded = files.upload()

        for filename in uploaded.keys():
            if filename.endswith('.csv'):
                po_df = pd.read_csv(io.BytesIO(uploaded[filename]))
                print(f"✅ PO data loaded from: {filename}")
                return po_df
    except:
        print("⚠️ No PO CSV uploaded. Generating sample data...")
        # Generate synthetic PO data
        np.random.seed(42)
        n_records = 500

        dates = pd.date_range('2023-01-01', '2024-12-31', periods=n_records)
        supplier_codes = [f'H{str(i).zfill(3)}' for i in range(1, 11)]

        po_data = {
            'po_number': [f'PO-{str(i).zfill(6)}' for i in range(1, n_records+1)],
            'gr_number': [f'GR-{str(i).zfill(6)}' for i in range(1, n_records+1)],
            'gr_date': dates.strftime('%Y-%m-%d').tolist(),
            'supplier_code': np.random.choice(supplier_codes, n_records),
            'ordered_qty': np.random.randint(1000, 20000, n_records),
            'received_qty': None,  # Will calculate based on fulfillment
            'rate': np.random.uniform(100, 1000, n_records).round(2),
            'item_value': None,  # Will calculate
            'moved_value': None,  # Will calculate
            'delivery_status': None  # Will assign
        }

        po_df = pd.DataFrame(po_data)

        # Calculate derived fields
        for idx, row in po_df.iterrows():
            # Generate realistic fulfillment (85-105%)
            fulfillment = np.random.normal(0.95, 0.05)
            fulfillment = max(0.85, min(1.05, fulfillment))

            po_df.at[idx, 'received_qty'] = int(row['ordered_qty'] * fulfillment)
            po_df.at[idx, 'item_value'] = row['ordered_qty'] * row['rate']

            # Generate moved value (with some discrepancies)
            discrepancy = np.random.normal(0.98, 0.02)
            po_df.at[idx, 'moved_value'] = row['item_value'] * discrepancy

            # Assign delivery status
            if fulfillment >= 0.98:
                po_df.at[idx, 'delivery_status'] = 'COMPLETE'
            elif fulfillment >= 0.90:
                po_df.at[idx, 'delivery_status'] = 'PARTIAL'
            else:
                po_df.at[idx, 'delivery_status'] = 'DELAYED'

    return po_df

# Load data
print("=" * 60)
print("📊 DATA LOADING MODULE")
print("=" * 60)

vendor_df = load_vendor_data()
po_df = load_po_data()

print(f"\n📈 Data Summary:")
print(f"   Vendors: {len(vendor_df)}")
print(f"   Purchase Orders: {len(po_df)}")
print(f"   Date Range: {po_df['gr_date'].min()} to {po_df['gr_date'].max()}")

# Display sample data
print("\n📋 Vendor Data Sample:")
display(vendor_df.head())

print("\n📋 Purchase Order Data Sample:")
display(po_df.head())

# 3. DATA QUALITY CHECK & PREPROCESSING


class DataQualityChecker:
    """Comprehensive data quality assessment and preprocessing"""

    def __init__(self, vendor_df, po_df):
        self.vendor_df = vendor_df
        self.po_df = po_df
        self.quality_report = {}

    def run_quality_checks(self):
        """Run all quality checks"""
        print("🔍 RUNNING DATA QUALITY CHECKS...")

        checks = [
            self.check_missing_values,
            self.check_data_types,
            self.check_duplicates,
            self.check_value_ranges,
            self.check_date_consistency,
            self.check_vendor_references
        ]

        for check in checks:
            check()

        self.generate_quality_report()

    def check_missing_values(self):
        """Check for missing values in all columns"""
        print("\n📊 Missing Values Analysis:")

        for df_name, df in [('Vendor', self.vendor_df), ('Purchase Orders', self.po_df)]:
            missing = df.isnull().sum()
            total = len(df)
            missing_pct = (missing / total * 100).round(2)

            missing_df = pd.DataFrame({
                'Column': missing.index,
                'Missing_Count': missing.values,
                'Missing_%': missing_pct.values
            }).sort_values('Missing_%', ascending=False)

            missing_df = missing_df[missing_df['Missing_Count'] > 0]

            if len(missing_df) > 0:
                print(f"\n❌ {df_name} Data - Missing Values Found:")
                display(missing_df)

                # Visualize missing values
                plt.figure(figsize=(10, 4))
                msno.matrix(df)
                plt.title(f'Missing Values in {df_name} Data', fontweight='bold')
                plt.show()
            else:
                print(f"✅ {df_name} Data: No missing values found")

    def check_data_types(self):
        """Verify data types are correct"""
        print("\n🔧 Data Types Validation:")

        for df_name, df in [('Vendor', self.vendor_df), ('Purchase Orders', self.po_df)]:
            print(f"\n{df_name} Data Types:")
            dtypes_df = pd.DataFrame({
                'Column': df.columns,
                'Data Type': df.dtypes.values,
                'Unique Values': [df[col].nunique() for col in df.columns]
            })
            display(dtypes_df)

    def check_duplicates(self):
        """Check for duplicate records"""
        print("\n🔍 Duplicate Records Check:")

        vendor_dups = self.vendor_df.duplicated().sum()
        po_dups = self.po_df.duplicated().sum()

        print(f"Vendor Master Duplicates: {vendor_dups}")
        print(f"Purchase Order Duplicates: {po_dups}")

        if vendor_dups > 0 or po_dups > 0:
            print("⚠️ Warning: Duplicates found. Consider cleaning data.")

    def check_value_ranges(self):
        """Check for outliers and invalid values"""
        print("\n📈 Value Range Analysis:")

        numeric_cols = self.po_df.select_dtypes(include=[np.number]).columns

        if len(numeric_cols) > 0:
            stats_df = pd.DataFrame()
            for col in numeric_cols:
                stats_df[col] = [
                    self.po_df[col].min(),
                    self.po_df[col].max(),
                    self.po_df[col].mean(),
                    self.po_df[col].median(),
                    self.po_df[col].std()
                ]

            stats_df.index = ['Min', 'Max', 'Mean', 'Median', 'Std Dev']
            display(stats_df.round(2))

            # Visualize distributions
            fig, axes = plt.subplots(2, 3, figsize=(15, 8))
            axes = axes.ravel()

            for idx, col in enumerate(numeric_cols[:6]):
                axes[idx].hist(self.po_df[col].dropna(), bins=30, alpha=0.7, color='steelblue')
                axes[idx].set_title(f'Distribution of {col}')
                axes[idx].set_xlabel(col)
                axes[idx].set_ylabel('Frequency')

            plt.suptitle('Numeric Value Distributions', fontsize=16, fontweight='bold')
            plt.tight_layout()
            plt.show()

    def check_date_consistency(self):
        """Check date ranges and consistency"""
        print("\n📅 Date Consistency Check:")

        if 'gr_date' in self.po_df.columns:
            self.po_df['gr_date'] = pd.to_datetime(self.po_df['gr_date'], format='%d-%m-%Y', errors='coerce')

            date_stats = {
                'Earliest Date': self.po_df['gr_date'].min(),
                'Latest Date': self.po_df['gr_date'].max(),
                'Date Range (days)': (self.po_df['gr_date'].max() - self.po_df['gr_date'].min()).days,
                'Records per Month': self.po_df['gr_date'].dt.to_period('M').value_counts().mean()
            }

            date_df = pd.DataFrame(list(date_stats.items()), columns=['Metric', 'Value'])
            display(date_df)

            # Monthly trend
            monthly_counts = self.po_df['gr_date'].dt.to_period('M').value_counts().sort_index()

            plt.figure(figsize=(12, 5))
            monthly_counts.plot(kind='bar', color='steelblue', alpha=0.7)
            plt.title('Purchase Orders by Month', fontsize=14, fontweight='bold')
            plt.xlabel('Month')
            plt.ylabel('Number of Orders')
            plt.xticks(rotation=45)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.show()
    def check_vendor_references(self):
        """Check if all PO vendors exist in vendor master"""
        print("\n🔗 Vendor Reference Integrity:")

        po_vendors = set(self.po_df['supplier_code'].unique())
        master_vendors = set(self.vendor_df['supplier_code'].unique())

        missing_in_master = po_vendors - master_vendors
        missing_in_po = master_vendors - po_vendors

        print(f"Vendors in POs but not in Master: {len(missing_in_master)}")
        if missing_in_master:
            print(f"   Codes: {list(missing_in_master)[:10]}...")

        print(f"Vendors in Master but not in POs: {len(missing_in_po)}")
        if missing_in_po:
            print(f"   Codes: {list(missing_in_po)[:10]}...")

        if len(missing_in_master) == 0:
            print("✅ All PO vendors exist in vendor master")

    def generate_quality_report(self):
        """Generate comprehensive data quality report"""
        self.quality_report = {
            'vendor_records': len(self.vendor_df),
            'po_records': len(self.po_df),
            'vendor_columns': len(self.vendor_df.columns),
            'po_columns': len(self.po_df.columns),
            'data_quality_score': 95.0  # Placeholder, would calculate based on checks
        }

        print("\n" + "=" * 60)
        print("📋 DATA QUALITY REPORT SUMMARY")
        print("=" * 60)
        for key, value in self.quality_report.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
        print("=" * 60)

        return self.quality_report

# Run data quality checks
print("\n" + "=" * 60)
print("🔍 DATA QUALITY ASSESSMENT")
print("=" * 60)

quality_checker = DataQualityChecker(vendor_df, po_df)
quality_report = quality_checker.run_quality_checks()

#4. SYNTHETIC DATA GENERATION FOR MISSING DIMENSIONS


class DataEnhancer:
    """Generate realistic synthetic data for missing analytical dimensions"""

    def __init__(self, vendor_df, po_df, random_seed=42):
        self.vendor_df = vendor_df
        self.po_df = po_df
        self.enhanced_data = None
        np.random.seed(random_seed)

        # Define vendor performance profiles
        self.vendor_profiles = {
            'high_performer': {'quality_mean': 95, 'quality_std': 2, 'rejection_rate': 0.5},
            'medium_performer': {'quality_mean': 88, 'quality_std': 3, 'rejection_rate': 2.0},
            'low_performer': {'quality_mean': 80, 'quality_std': 5, 'rejection_rate': 5.0}
        }

        # Assign performance tiers to vendors (simulated)
        self.assign_vendor_tiers()

    def assign_vendor_tiers(self):
        """Assign random performance tiers to vendors"""
        np.random.seed(42)
        n_vendors = len(self.vendor_df)

        # Assign tiers (30% high, 50% medium, 20% low)
        tiers = np.random.choice(
            ['high_performer', 'medium_performer', 'low_performer'],
            size=n_vendors,
            p=[0.3, 0.5, 0.2]
        )

        self.vendor_df['performance_tier'] = tiers

        # Map supplier codes to tiers
        self.vendor_tier_map = dict(zip(
            self.vendor_df['supplier_code'],
            self.vendor_df['performance_tier']
        ))

    def generate_quality_metrics(self):
        """Generate quality scores and rejection quantities"""
        print("🔧 Generating Quality Metrics...")

        quality_data = []

        for idx, po in self.po_df.iterrows():
            vendor_code = po['supplier_code']
            received_qty = po['received_qty']

            # Get vendor's performance profile
            tier = self.vendor_tier_map.get(vendor_code, 'medium_performer')
            profile = self.vendor_profiles[tier]

            # Generate quality score
            quality_score = np.random.normal(profile['quality_mean'], profile['quality_std'])
            quality_score = np.clip(quality_score, 70, 100)

            # Generate rejection quantity
            base_rejection = profile['rejection_rate'] / 100
            variance = np.random.uniform(0.5, 1.5)
            rejection_pct = base_rejection * variance
            rejected_qty = received_qty * rejection_pct

            quality_data.append({
                'po_number': po['po_number'],
                'quality_score': round(quality_score, 1),
                'rejected_quantity': round(rejected_qty, 2),
                'accepted_quantity': round(received_qty - rejected_qty, 2),
                'quality_status': 'ACCEPTED' if quality_score >= 85 else 'REVIEW_NEEDED'
            })

        return pd.DataFrame(quality_data)

    def generate_delivery_metrics(self):
        """Generate delivery timing and reliability metrics"""
        print("🔧 Generating Delivery Metrics...")

        delivery_data = []

        for idx, po in self.po_df.iterrows():
            vendor_code = po['supplier_code']
            gr_date = pd.to_datetime(po['gr_date'], format='%d-%m-%Y')

            # Get vendor's performance profile
            tier = self.vendor_tier_map.get(vendor_code, 'medium_performer')

            # Different lead times based on tier
            if tier == 'high_performer':
                lead_time = np.random.normal(10, 2)
                on_time_prob = 0.95
            elif tier == 'medium_performer':
                lead_time = np.random.normal(15, 3)
                on_time_prob = 0.85
            else:
                lead_time = np.random.normal(20, 5)
                on_time_prob = 0.70

            # Generate order date
            order_date = gr_date - timedelta(days=int(lead_time))

            # Generate expected delivery (90% of lead time)
            expected_days = int(lead_time * 0.9)
            expected_delivery = order_date + timedelta(days=expected_days)

            # Determine if delivery was on time
            is_ontime = np.random.random() < on_time_prob

            if is_ontime:
                actual_delivery = expected_delivery + timedelta(days=np.random.randint(-2, 1))
                delay_days = 0
            else:
                delay = np.random.exponential(7) + 1  # Minimum 1 day delay
                actual_delivery = expected_delivery + timedelta(days=int(delay))
                delay_days = int(delay)

            # Calculate lead time
            total_lead_time = (actual_delivery - order_date).days

            delivery_data.append({
                'po_number': po['po_number'],
                'order_date': order_date.strftime('%Y-%m-%d'),
                'expected_delivery': expected_delivery.strftime('%Y-%m-%d'),
                'actual_delivery': actual_delivery.strftime('%Y-%m-%d'),
                'delay_days': delay_days,
                'lead_time_days': total_lead_time,
                'is_ontime': is_ontime,
                'on_time_status': 'ONTIME' if is_ontime else 'DELAYED'
            })

        return pd.DataFrame(delivery_data)

    def generate_responsiveness_metrics(self):
        """Generate responsiveness and communication metrics"""
        print("🔧 Generating Responsiveness Metrics...")

        response_data = []

        for idx, po in self.po_df.iterrows():
            vendor_code = po['supplier_code']
            tier = self.vendor_tier_map.get(vendor_code, 'medium_performer')

            # Response times based on tier
            if tier == 'high_performer':
                response_hours = np.random.exponential(2)
                urgent_acceptance = 0.98
                email_response = np.random.exponential(4)
            elif tier == 'medium_performer':
                response_hours = np.random.exponential(6)
                urgent_acceptance = 0.85
                email_response = np.random.exponential(8)
            else:
                response_hours = np.random.exponential(12)
                urgent_acceptance = 0.65
                email_response = np.random.exponential(24)

            # 20% of orders are urgent
            is_urgent = np.random.random() < 0.20
            accepted = np.random.random() < urgent_acceptance if is_urgent else True

            response_data.append({
                'po_number': po['po_number'],
                'response_time_hours': round(response_hours, 1),
                'email_response_hours': round(email_response, 1),
                'is_urgent_order': is_urgent,
                'urgent_order_accepted': accepted,
                'query_resolution_days': np.random.exponential(2) + 1,
                'communication_score': np.random.normal(85 if tier == 'high_performer' else 70, 10)
            })

        response_df = pd.DataFrame(response_data)
        response_df['communication_score'] = response_df['communication_score'].clip(50, 100).round(1)

        return response_df

    def generate_invoice_metrics(self):
        """Generate invoice accuracy and compliance metrics"""
        print("🔧 Generating Invoice Metrics...")

        invoice_data = []

        for idx, po in self.po_df.iterrows():
            vendor_code = po['supplier_code']
            tier = self.vendor_tier_map.get(vendor_code, 'medium_performer')

            if tier == 'high_performer':
                error_prob = 0.01
                term_compliance = 0.98
                doc_completeness = 0.99
            elif tier == 'medium_performer':
                error_prob = 0.05
                term_compliance = 0.90
                doc_completeness = 0.85
            else:
                error_prob = 0.15
                term_compliance = 0.70
                doc_completeness = 0.60

            has_error = np.random.random() < error_prob
            terms_match = np.random.random() < term_compliance
            docs_complete = np.random.random() < doc_completeness

            # Calculate invoice accuracy score
            accuracy_score = 100
            if has_error:
                accuracy_score -= 20
            if not terms_match:
                accuracy_score -= 15
            if not docs_complete:
                accuracy_score -= 10

            invoice_data.append({
                'po_number': po['po_number'],
                'invoice_has_error': has_error,
                'payment_terms_match': terms_match,
                'documentation_complete': docs_complete,
                'invoice_accuracy_score': round(accuracy_score, 1),
                'dispute_resolution_days': np.random.exponential(5) + 2 if has_error else 0
            })

        return pd.DataFrame(invoice_data)

    def generate_sustainability_metrics(self):
        """Generate sustainability and compliance metrics"""
        print("🔧 Generating Sustainability Metrics...")

        sustainability_data = []

        for idx, vendor in self.vendor_df.iterrows():
            vendor_code = vendor['supplier_code']
            tier = vendor.get('performance_tier', 'medium_performer')

            # Assign certifications based on tier
            if tier == 'high_performer':
                certifications = np.random.choice(['ISO14001', 'ISO9001', 'OHSAS18001', 'R2', 'e-Stewards'],
                                                  size=np.random.randint(2, 5), replace=False)
                has_eco_vat = True
                violation_count = np.random.poisson(0.1)
            elif tier == 'medium_performer':
                certifications = np.random.choice(['ISO9001', 'ISO14001', 'OHSAS18001'],
                                                  size=np.random.randint(1, 3), replace=False)
                has_eco_vat = np.random.random() < 0.5
                violation_count = np.random.poisson(0.5)
            else:
                certifications = ['ISO9001'] if np.random.random() < 0.3 else []
                has_eco_vat = False
                violation_count = np.random.poisson(1.5)

            sustainability_data.append({
                'supplier_code': vendor_code,
                'certifications': ','.join(certifications) if len(certifications) > 0 else 'None',
                'num_certifications': len(certifications),
                'has_eco_vat': has_eco_vat,
                'sustainability_violations': int(violation_count),
                'carbon_footprint_score': np.random.normal(75 if tier == 'high_performer' else 60, 10),
                'renewable_energy_pct': np.random.uniform(20 if tier == 'low_performer' else 60, 100)
            })

        sustainability_df = pd.DataFrame(sustainability_data)
        sustainability_df['carbon_footprint_score'] = sustainability_df['carbon_footprint_score'].clip(40, 100).round(1)
        sustainability_df['renewable_energy_pct'] = sustainability_df['renewable_energy_pct'].round(1)

        return sustainability_df

    def enhance_dataset(self):
        """Generate all synthetic metrics and enhance dataset"""
        print("\n" + "=" * 60)
        print("🔧 ENHANCING DATASET WITH SYNTHETIC METRICS")
        print("=" * 60)

        # Generate all metrics
        quality_df = self.generate_quality_metrics()
        delivery_df = self.generate_delivery_metrics()
        response_df = self.generate_responsiveness_metrics()
        invoice_df = self.generate_invoice_metrics()
        sustainability_df = self.generate_sustainability_metrics()

        # Start with PO data
        enhanced_data = self.po_df.copy()

        # Merge all metrics
        enhanced_data = enhanced_data.merge(quality_df, on='po_number', how='left')
        enhanced_data = enhanced_data.merge(delivery_df, on='po_number', how='left')
        enhanced_data = enhanced_data.merge(response_df, on='po_number', how='left')
        enhanced_data = enhanced_data.merge(invoice_df, on='po_number', how='left')

        # Merge vendor info
        enhanced_data = enhanced_data.merge(self.vendor_df, on='supplier_code', how='left')

        # Merge sustainability (vendor-level)
        enhanced_data = enhanced_data.merge(sustainability_df, on='supplier_code', how='left')

        # Calculate derived metrics
        enhanced_data['fulfillment_rate'] = (
            enhanced_data['received_qty'] / enhanced_data['ordered_qty'] * 100
        ).round(2)

        enhanced_data['value_discrepancy_pct'] = (
            (enhanced_data['item_value'] - enhanced_data['moved_value']).abs() /
            enhanced_data['item_value'] * 100
        ).round(2)

        enhanced_data['unit_price_variance'] = (
            enhanced_data.groupby('supplier_code')['rate'].transform('std') /
            enhanced_data.groupby('supplier_code')['rate'].transform('mean') * 100
        ).round(2)

        self.enhanced_data = enhanced_data

        print(f"✅ Dataset enhanced with {len(enhanced_data.columns)} columns")
        print(f"   Total records: {len(enhanced_data)}")
        print(f"   Synthetic features added: {len(enhanced_data.columns) - len(self.po_df.columns)}")

        return enhanced_data

# Enhance dataset
print("\n" + "=" * 60)
print("🔧 DATA ENHANCEMENT PROCESS")
print("=" * 60)

enhancer = DataEnhancer(vendor_df, po_df)
enhanced_df = enhancer.enhance_dataset()

print("\n📊 Enhanced Dataset Sample:")
display(enhanced_df.head())

print("\n📋 Enhanced Dataset Columns:")
print(enhanced_df.columns.tolist())

# 5. VENDOR SCORING ENGINE


class VendorScoringEngine:
    """Multi-dimensional vendor performance scoring system"""

    def __init__(self, enhanced_df, weights=None):
        self.df = enhanced_df
        self.vendor_scores = None

        # Default weights for each dimension
        self.weights = weights or {
            'quality': 0.25,      # Quality consistency
            'delivery': 0.25,     # Delivery reliability
            'price': 0.20,        # Price competitiveness
            'responsiveness': 0.15, # Responsiveness
            'invoice': 0.10,      # Invoice accuracy
            'sustainability': 0.05 # Sustainability
        }

        # Validate weights sum to 1
        weight_sum = sum(self.weights.values())
        if abs(weight_sum - 1.0) > 0.01:
            raise ValueError(f"Weights must sum to 1.0, got {weight_sum}")

    def calculate_quality_scores(self):
        """Calculate quality dimension scores"""
        print("📊 Calculating Quality Scores...")

        quality_metrics = self.df.groupby('supplier_code').agg({
            'quality_score': ['mean', 'std', 'count'],
            'rejected_quantity': 'sum',
            'received_qty': 'sum'
        }).round(3)

        quality_metrics.columns = ['quality_mean', 'quality_std', 'order_count',
                                  'total_rejected', 'total_received']

        # Calculate derived metrics
        quality_metrics['rejection_rate'] = (
            quality_metrics['total_rejected'] / quality_metrics['total_received'] * 100
        ).round(2)

        quality_metrics['quality_consistency'] = (
            100 - quality_metrics['quality_std'].fillna(0) * 10
        ).clip(0, 100).round(1)

        # Calculate quality sub-score
        quality_metrics['quality_subscore'] = (
            (100 - np.clip(quality_metrics['rejection_rate'], 0, 20) * 2.5) * 0.4 +
            quality_metrics['quality_mean'] * 0.4 +
            quality_metrics['quality_consistency'] * 0.2
        ).round(1)

        return quality_metrics[['quality_mean', 'rejection_rate',
                               'quality_consistency', 'quality_subscore']]

    def calculate_delivery_scores(self):
        """Calculate delivery dimension scores"""
        print("📊 Calculating Delivery Scores...")

        delivery_metrics = self.df.groupby('supplier_code').agg({
            'is_ontime': 'mean',
            'delay_days': lambda x: x[x > 0].mean(),
            'fulfillment_rate': 'mean',
            'delivery_status': lambda x: (x == 'COMPLETE').mean(),
            'lead_time_days': ['mean', 'std']
        }).round(3)

        delivery_metrics.columns = ['ontime_rate', 'avg_delay', 'fulfillment_rate',
                                   'complete_rate', 'avg_lead_time', 'lead_time_std']

        # Calculate delivery consistency
        delivery_metrics['lead_time_consistency'] = (
            100 - delivery_metrics['lead_time_std'].fillna(0) * 5
        ).clip(0, 100).round(1)

        # Calculate delivery sub-score
        delivery_metrics['delivery_subscore'] = (
            delivery_metrics['ontime_rate'] * 100 * 0.25 +
            (100 - np.clip(delivery_metrics['avg_delay'].fillna(0), 0, 30)) * 0.25 +
            delivery_metrics['fulfillment_rate'] * 0.25 +
            delivery_metrics['lead_time_consistency'] * 0.25
        ).round(1)

        return delivery_metrics[['ontime_rate', 'avg_delay', 'fulfillment_rate',
                                'lead_time_consistency', 'delivery_subscore']]

    def calculate_price_scores(self):
        """Calculate price dimension scores"""
        print("📊 Calculating Price Scores...")

        # Calculate category median prices
        if 'vendor_category' in self.df.columns:
            category_median = self.df.groupby('vendor_category')['rate'].median()

            price_metrics = self.df.groupby('supplier_code').agg({
                'rate': ['mean', 'std', 'count'],
                'vendor_category': 'first'
            }).round(3)

            price_metrics.columns = ['avg_price', 'price_std', 'order_count', 'category']

            price_metrics['category_median'] = price_metrics['category'].map(category_median)
        else:
            overall_median = self.df['rate'].median()

            price_metrics = self.df.groupby('supplier_code').agg({
                'rate': ['mean', 'std', 'count']
            }).round(3)

            price_metrics.columns = ['avg_price', 'price_std', 'order_count']
            price_metrics['category_median'] = overall_median

        # Calculate price index relative to median
        price_metrics['price_index'] = (
            price_metrics['avg_price'] / price_metrics['category_median'] * 100
        ).round(1)

        # Calculate price stability
        price_metrics['price_stability'] = (
            100 - (price_metrics['price_std'] / price_metrics['avg_price'] * 100)
        ).clip(0, 100).round(1)

        # Calculate price sub-score (lower price index is better)
        price_deviation = (price_metrics['price_index'] - 100).clip(0, 50)
        price_metrics['price_subscore'] = (
            (100 - price_deviation * 1.5) * 0.6 +
            price_metrics['price_stability'] * 0.4
        ).round(1)

        return price_metrics[['avg_price', 'price_index', 'price_stability', 'price_subscore']]

    def calculate_responsiveness_scores(self):
        """Calculate responsiveness dimension scores"""
        print("📊 Calculating Responsiveness Scores...")

        # Filter urgent orders
        urgent_orders = self.df[self.df['is_urgent_order']]

        if len(urgent_orders) > 0:
            urgent_metrics = urgent_orders.groupby('supplier_code').agg({
                'urgent_order_accepted': 'mean',
                'response_time_hours': 'mean',
                'email_response_hours': 'mean',
                'communication_score': 'mean'
            }).round(3)
            urgent_metrics.columns = ['urgent_acceptance', 'avg_response_time',
                                     'avg_email_response', 'avg_communication_score']
        else:
            urgent_metrics = pd.DataFrame()

        # Calculate overall response metrics
        response_metrics = self.df.groupby('supplier_code').agg({
            'query_resolution_days': 'mean',
            'communication_score': 'mean'
        }).round(3)

        response_metrics.columns = ['avg_query_resolution', 'communication_score']

        # Combine metrics
        all_metrics = pd.DataFrame(index=self.df['supplier_code'].unique())

        for metric in ['urgent_acceptance', 'avg_response_time', 'avg_email_response']:
            if metric in urgent_metrics.columns:
                all_metrics[metric] = urgent_metrics[metric]
            else:
                all_metrics[metric] = 0.8 if metric == 'urgent_acceptance' else 6.0

        all_metrics['avg_query_resolution'] = response_metrics['avg_query_resolution']
        all_metrics['communication_score'] = response_metrics['communication_score'].fillna(70)

        # Calculate responsiveness sub-score
        all_metrics['response_subscore'] = (
            all_metrics['urgent_acceptance'] * 100 * 0.25 +
            (100 - np.clip(all_metrics['avg_response_time'] * 5, 0, 100)) * 0.25 +
            (100 - np.clip(all_metrics['avg_query_resolution'] * 10, 0, 100)) * 0.25 +
            all_metrics['communication_score'] * 0.25
        ).round(1)

        return all_metrics[['urgent_acceptance', 'avg_response_time',
                           'avg_query_resolution', 'communication_score', 'response_subscore']]

    def calculate_invoice_scores(self):
        """Calculate invoice accuracy dimension scores"""
        print("📊 Calculating Invoice Scores...")

        invoice_metrics = self.df.groupby('supplier_code').agg({
            'invoice_has_error': 'mean',
            'payment_terms_match': 'mean',
            'documentation_complete': 'mean',
            'invoice_accuracy_score': 'mean',
            'dispute_resolution_days': lambda x: x[x > 0].mean()
        }).round(3)

        invoice_metrics.columns = ['error_rate', 'terms_compliance', 'doc_completeness',
                                  'accuracy_score', 'avg_dispute_resolution']

        # Calculate invoice sub-score
        invoice_metrics['invoice_subscore'] = (
            invoice_metrics['accuracy_score'] * 0.4 +
            (100 - invoice_metrics['error_rate'] * 100) * 0.3 +
            invoice_metrics['terms_compliance'] * 100 * 0.2 +
            invoice_metrics['doc_completeness'] * 100 * 0.1
        ).round(1)

        return invoice_metrics[['error_rate', 'terms_compliance', 'doc_completeness',
                               'accuracy_score', 'invoice_subscore']]

    def calculate_sustainability_scores(self):
        """Calculate sustainability dimension scores"""
        print("📊 Calculating Sustainability Scores...")

        sustainability_metrics = self.df.groupby('supplier_code').agg({
            'num_certifications': 'first',
            'has_eco_vat': 'first',
            'sustainability_violations': 'first',
            'carbon_footprint_score': 'first',
            'renewable_energy_pct': 'first'
        })

        # Calculate sustainability sub-score
        sustainability_metrics['certification_score'] = (
            sustainability_metrics['num_certifications'] * 20
        ).clip(0, 100)

        sustainability_metrics['eco_vat_score'] = (
            sustainability_metrics['has_eco_vat'].astype(int) * 30
        )

        sustainability_metrics['violation_deduction'] = (
            sustainability_metrics['sustainability_violations'] * 10
        ).clip(0, 50)

        sustainability_metrics['sustainability_subscore'] = (
            sustainability_metrics['certification_score'] * 0.3 +
            sustainability_metrics['eco_vat_score'] * 0.2 +
            sustainability_metrics['carbon_footprint_score'] * 0.3 +
            sustainability_metrics['renewable_energy_pct'] * 0.2 -
            sustainability_metrics['violation_deduction']
        ).clip(0, 100).round(1)

        return sustainability_metrics[['num_certifications', 'sustainability_violations',
                                      'carbon_footprint_score', 'renewable_energy_pct',
                                      'sustainability_subscore']]

    def calculate_composite_scores(self):
        """Calculate all scores and composite performance"""
        print("\n" + "=" * 60)
        print("🎯 CALCULATING COMPOSITE VENDOR SCORES")
        print("=" * 60)

        # Calculate all dimension scores
        quality_scores = self.calculate_quality_scores()
        delivery_scores = self.calculate_delivery_scores()
        price_scores = self.calculate_price_scores()
        response_scores = self.calculate_responsiveness_scores()
        invoice_scores = self.calculate_invoice_scores()
        sustainability_scores = self.calculate_sustainability_scores()

        # Combine all scores
        vendor_scores = pd.concat([
            quality_scores,
            delivery_scores,
            price_scores,
            response_scores,
            invoice_scores,
            sustainability_scores
        ], axis=1)

        # Add vendor information
        vendor_info = self.df.groupby('supplier_code').agg({
            'vendor_name': 'first',
            'vendor_category': 'first',
            'vendor_location': 'first',
            'payment_terms': 'first'
        })

        vendor_scores = vendor_scores.join(vendor_info, how='left')

        # Calculate weighted composite score
        vendor_scores['composite_score'] = (
            vendor_scores['quality_subscore'] * self.weights['quality'] +
            vendor_scores['delivery_subscore'] * self.weights['delivery'] +
            vendor_scores['price_subscore'] * self.weights['price'] +
            vendor_scores['response_subscore'] * self.weights['responsiveness'] +
            vendor_scores['invoice_subscore'] * self.weights['invoice'] +
            vendor_scores['sustainability_subscore'] * self.weights['sustainability']
        ).round(1)

        # Assign vendor tiers
        vendor_scores['tier'] = vendor_scores['composite_score'].apply(
            lambda x: 'Tier 1 - Strategic' if x >= 85 else
                     ('Tier 2 - Preferred' if x >= 70 else 'Tier 3 - Transactional')
        )

        # Calculate ranks
        vendor_scores['overall_rank'] = vendor_scores['composite_score'].rank(ascending=False, method='min').fillna(len(vendor_scores) + 1).astype(int)

        vendor_scores['category_rank'] = vendor_scores.groupby('vendor_category')['composite_score'].rank(ascending=False, method='min').fillna(999).astype(int)
        vendor_scores['composite_score'] = vendor_scores['composite_score'].fillna(0)
        # Calculate performance gaps
        vendor_scores['gap_to_next_tier'] = vendor_scores.apply(
            lambda x: 85 - x['composite_score'] if x['tier'] == 'Tier 2 - Preferred' else
                     (70 - x['composite_score'] if x['tier'] == 'Tier 3 - Transactional' else 0),
            axis=1
        )

        # Calculate risk score (inverse of composite score)
        vendor_scores['risk_score'] = (100 - vendor_scores['composite_score']).round(1)

        self.vendor_scores = vendor_scores

        print(f"✅ Scores calculated for {len(vendor_scores)} vendors")
        print(f"📊 Tier Distribution:")
        print(vendor_scores['tier'].value_counts())

        return vendor_scores

    def get_vendor_report(self, supplier_code):
        """Get detailed report for specific vendor"""
        if supplier_code not in self.vendor_scores.index:
            raise ValueError(f"Vendor {supplier_code} not found in scores")

        vendor_data = self.vendor_scores.loc[supplier_code]

        report = {
            'Vendor Information': {
                'Name': vendor_data['vendor_name'],
                'Category': vendor_data['vendor_category'],
                'Location': vendor_data['vendor_location'],
                'Payment Terms': vendor_data['payment_terms']
            },
            'Performance Scores': {
                'Composite Score': vendor_data['composite_score'],
                'Tier': vendor_data['tier'],
                'Overall Rank': f"#{vendor_data['overall_rank']}",
                'Category Rank': f"#{vendor_data['category_rank']}"
            },
            'Dimension Scores': {
                'Quality': vendor_data['quality_subscore'],
                'Delivery': vendor_data['delivery_subscore'],
                'Price': vendor_data['price_subscore'],
                'Responsiveness': vendor_data['response_subscore'],
                'Invoice': vendor_data['invoice_subscore'],
                'Sustainability': vendor_data['sustainability_subscore']
            },
            'Key Metrics': {
                'Rejection Rate': f"{vendor_data['rejection_rate']}%",
                'On-Time Delivery': f"{vendor_data['ontime_rate']*100:.1f}%",
                'Price Index': vendor_data['price_index'],
                'Communication Score': vendor_data['communication_score']
            }
        }

        return report

# Run scoring engine
print("\n" + "=" * 60)
print("🎯 VENDOR SCORING ENGINE")
print("=" * 60)

scoring_engine = VendorScoringEngine(enhanced_df)
vendor_scores = scoring_engine.calculate_composite_scores()

print("\n📊 Vendor Scores Summary:")
display(vendor_scores.head())

# Get sample vendor report
sample_vendor = vendor_scores.index[0]
print(f"\n📋 Sample Report for Vendor: {sample_vendor}")
vendor_report = scoring_engine.get_vendor_report(sample_vendor)

for category, data in vendor_report.items():
    print(f"\n{category}:")
    for key, value in data.items():
        print(f"  {key}: {value}")

#6. ADVANCED ANALYTICS & VISUALIZATIONS

class AdvancedAnalytics:
    """Advanced analytics and visualization suite"""

    def __init__(self, vendor_scores, enhanced_df):
        self.vendor_scores = vendor_scores
        self.enhanced_df = enhanced_df

    def create_comprehensive_dashboard(self):
        """Create comprehensive visualization dashboard"""
        print("\n" + "=" * 60)
        print("📊 CREATING ANALYTICS DASHBOARD")
        print("=" * 60)

        # Create subplots
        fig = plt.figure(figsize=(20, 16))

        # 1. Tier Distribution (Pie Chart)
        ax1 = plt.subplot(3, 3, 1)
        tier_counts = self.vendor_scores['tier'].value_counts()
        colors = ['#2E7D32', '#FF9800', '#D32F2F']
        ax1.pie(tier_counts.values, labels=tier_counts.index, autopct='%1.1f%%',
               colors=colors, startangle=90)
        ax1.set_title('Vendor Tier Distribution', fontsize=14, fontweight='bold')

        # 2. Score Distribution (Histogram)
        ax2 = plt.subplot(3, 3, 2)
        ax2.hist(self.vendor_scores['composite_score'], bins=20, alpha=0.7,
                color='steelblue', edgecolor='black')
        ax2.axvline(x=85, color='green', linestyle='--', alpha=0.7, label='Tier 1 Threshold')
        ax2.axvline(x=70, color='orange', linestyle='--', alpha=0.7, label='Tier 2 Threshold')
        ax2.set_xlabel('Composite Score')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Composite Score Distribution', fontsize=14, fontweight='bold')
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        # 3. Category Performance (Bar Chart)
        ax3 = plt.subplot(3, 3, 3)
        category_performance = self.vendor_scores.groupby('vendor_category')['composite_score'].mean().sort_values()
        category_performance.plot(kind='barh', ax=ax3, color='teal', alpha=0.7)
        ax3.set_xlabel('Average Composite Score')
        ax3.set_title('Performance by Category', fontsize=14, fontweight='bold')
        ax3.grid(True, alpha=0.3)

        # 4. Geographic Performance (Bar Chart)
        ax4 = plt.subplot(3, 3, 4)
        location_performance = self.vendor_scores.groupby('vendor_location')['composite_score'].mean().sort_values()
        location_performance.plot(kind='bar', ax=ax4, color='purple', alpha=0.7)
        ax4.set_xlabel('Location')
        ax4.set_ylabel('Average Score')
        ax4.set_title('Performance by Location', fontsize=14, fontweight='bold')
        ax4.tick_params(axis='x', rotation=45)
        ax4.grid(True, alpha=0.3)

        # 5. Dimension Correlation Heatmap
        ax5 = plt.subplot(3, 3, 5)
        dimensions = ['quality_subscore', 'delivery_subscore', 'price_subscore',
                     'response_subscore', 'invoice_subscore', 'sustainability_subscore']
        correlation_matrix = self.vendor_scores[dimensions].corr()

        im = ax5.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1)
        ax5.set_xticks(range(len(dimensions)))
        ax5.set_yticks(range(len(dimensions)))
        ax5.set_xticklabels([d.replace('_subscore', '') for d in dimensions], rotation=45)
        ax5.set_yticklabels([d.replace('_subscore', '') for d in dimensions])
        ax5.set_title('Dimension Correlation Matrix', fontsize=14, fontweight='bold')

        # Add correlation values
        for i in range(len(dimensions)):
            for j in range(len(dimensions)):
                text = ax5.text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}',
                              ha="center", va="center", color="white", fontsize=9)

        # 6. Top 10 Vendors (Bar Chart)
        ax6 = plt.subplot(3, 3, 6)
        top_10 = self.vendor_scores.nlargest(10, 'composite_score')
        bars = ax6.barh(range(len(top_10)), top_10['composite_score'],
                       color=['#2E7D32']*len(top_10), alpha=0.7)
        ax6.set_yticks(range(len(top_10)))
        ax6.set_yticklabels(top_10.index)
        ax6.set_xlabel('Composite Score')
        ax6.set_title('Top 10 Performing Vendors', fontsize=14, fontweight='bold')
        ax6.invert_yaxis()
        ax6.grid(True, alpha=0.3)

        # Add score values on bars
        for i, (idx, row) in enumerate(top_10.iterrows()):
            ax6.text(row['composite_score'] + 0.5, i, f'{row["composite_score"]}',
                    va='center', fontsize=9)

        # 7. Performance vs Price (Scatter Plot)
        ax7 = plt.subplot(3, 3, 7)
        scatter = ax7.scatter(self.vendor_scores['price_index'],
                             self.vendor_scores['composite_score'],
                             c=self.vendor_scores['composite_score'],
                             cmap='viridis', alpha=0.6, s=100)
        ax7.axvline(x=100, color='red', linestyle='--', alpha=0.5, label='Category Median')
        ax7.set_xlabel('Price Index (100 = Category Median)')
        ax7.set_ylabel('Composite Score')
        ax7.set_title('Performance vs Price Competitiveness', fontsize=14, fontweight='bold')
        ax7.legend()
        ax7.grid(True, alpha=0.3)
        plt.colorbar(scatter, ax=ax7, label='Composite Score')

        # 8. Delivery Performance Distribution
        ax8 = plt.subplot(3, 3, 8)
        delivery_scores = self.vendor_scores['delivery_subscore']
        ax8.hist(delivery_scores, bins=15, alpha=0.7, color='orange', edgecolor='black')
        ax8.axvline(x=delivery_scores.mean(), color='red', linestyle='--',
                   label=f'Mean: {delivery_scores.mean():.1f}')
        ax8.set_xlabel('Delivery Score')
        ax8.set_ylabel('Frequency')
        ax8.set_title('Delivery Performance Distribution', fontsize=14, fontweight='bold')
        ax8.legend()
        ax8.grid(True, alpha=0.3)

        # 9. Quality vs Delivery Correlation
        ax9 = plt.subplot(3, 3, 9)
        ax9.scatter(self.vendor_scores['quality_subscore'],
                   self.vendor_scores['delivery_subscore'],
                   c=self.vendor_scores['tier'].map({'Tier 1 - Strategic': '#2E7D32',
                                                    'Tier 2 - Preferred': '#FF9800',
                                                    'Tier 3 - Transactional': '#D32F2F'}),
                   alpha=0.6, s=100)
        ax9.set_xlabel('Quality Score')
        ax9.set_ylabel('Delivery Score')
        ax9.set_title('Quality vs Delivery Performance', fontsize=14, fontweight='bold')
        ax9.grid(True, alpha=0.3)

        # Add trend line
        z = np.polyfit(self.vendor_scores['quality_subscore'],
                      self.vendor_scores['delivery_subscore'], 1)
        p = np.poly1d(z)
        ax9.plot(self.vendor_scores['quality_subscore'],
                p(self.vendor_scores['quality_subscore']),
                "r--", alpha=0.8)

        plt.suptitle('Vendor Performance Analytics Dashboard',
                    fontsize=18, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()

    def create_radar_chart(self, vendor_codes=None):
        """Create radar chart for selected vendors"""
        if vendor_codes is None:
            vendor_codes = self.vendor_scores.nlargest(3, 'composite_score').index.tolist()

        dimensions = ['Quality', 'Delivery', 'Price', 'Responsiveness', 'Invoice', 'Sustainability']
        dim_mapping = {
            'Quality': 'quality_subscore',
            'Delivery': 'delivery_subscore',
            'Price': 'price_subscore',
            'Responsiveness': 'response_subscore',
            'Invoice': 'invoice_subscore',
            'Sustainability': 'sustainability_subscore'
        }

        fig = go.Figure()

        for vendor_code in vendor_codes:
            vendor_data = self.vendor_scores.loc[vendor_code]
            values = [vendor_data[dim_mapping[dim]] for dim in dimensions]
            values.append(values[0])  # Close the radar

            fig.add_trace(go.Scatterpolar(
                r=values,
                theta=dimensions + [dimensions[0]],
                name=f"{vendor_code}: {vendor_data['vendor_name']}",
                fill='toself',
                opacity=0.7
            ))

        fig.update_layout(
            polar=dict(
                radialaxis=dict(
                    visible=True,
                    range=[0, 100]
                )),
            showlegend=True,
            title={
                'text': 'Vendor Performance Radar Chart',
                'font': {'size': 16, 'weight': 'bold'}
            },
            height=600
        )

        fig.show()

    def create_trend_analysis(self):
        """Analyze performance trends over time"""
        print("\n📈 Performance Trend Analysis...")

        if 'gr_date' not in self.enhanced_df.columns:
            print("⚠️ Date information not available for trend analysis")
            return

        # Convert to datetime
        self.enhanced_df['gr_date'] = pd.to_datetime(self.enhanced_df['gr_date'])

        # Create monthly aggregates
        self.enhanced_df['year_month'] = self.enhanced_df['gr_date'].dt.to_period('M')

        monthly_metrics = self.enhanced_df.groupby(['supplier_code', 'year_month']).agg({
            'quality_score': 'mean',
            'fulfillment_rate': 'mean',
            'delay_days': 'mean',
            'is_ontime': 'mean'
        }).reset_index()

        # Pivot for visualization
        quality_trend = monthly_metrics.pivot(index='year_month',
                                             columns='supplier_code',
                                             values='quality_score')

        # Plot trends
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))

        # Quality trend for top 5 vendors
        top_5 = self.vendor_scores.nlargest(5, 'composite_score').index
        quality_top5 = quality_trend[top_5]

        for vendor in top_5:
            axes[0, 0].plot(quality_top5.index.astype(str), quality_top5[vendor],
                           marker='o', linewidth=2, label=vendor)

        axes[0, 0].set_title('Quality Score Trend (Top 5 Vendors)', fontsize=12, fontweight='bold')
        axes[0, 0].set_xlabel('Month')
        axes[0, 0].set_ylabel('Quality Score')
        axes[0, 0].legend(loc='best')
        axes[0, 0].tick_params(axis='x', rotation=45)
        axes[0, 0].grid(True, alpha=0.3)

        # On-time delivery trend
        ontime_trend = monthly_metrics.pivot(index='year_month',
                                            columns='supplier_code',
                                            values='is_ontime')

        for vendor in top_5:
            if vendor in ontime_trend.columns:
                axes[0, 1].plot(ontime_trend.index.astype(str), ontime_trend[vendor] * 100,
                               marker='s', linewidth=2, label=vendor)

        axes[0, 1].set_title('On-Time Delivery % Trend (Top 5 Vendors)', fontsize=12, fontweight='bold')
        axes[0, 1].set_xlabel('Month')
        axes[0, 1].set_ylabel('On-Time Delivery %')
        axes[0, 1].legend(loc='best')
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].grid(True, alpha=0.3)

        # Fulfillment rate distribution
        fulfillment_data = self.enhanced_df['fulfillment_rate']
        axes[1, 0].hist(fulfillment_data, bins=20, alpha=0.7, color='green', edgecolor='black')
        axes[1, 0].axvline(x=fulfillment_data.mean(), color='red', linestyle='--',
                          label=f'Mean: {fulfillment_data.mean():.1f}%')
        axes[1, 0].set_xlabel('Fulfillment Rate (%)')
        axes[1, 0].set_ylabel('Frequency')
        axes[1, 0].set_title('Fulfillment Rate Distribution', fontsize=12, fontweight='bold')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

        # Delay days distribution
        delay_data = self.enhanced_df[self.enhanced_df['delay_days'] > 0]['delay_days']
        axes[1, 1].hist(delay_data, bins=20, alpha=0.7, color='red', edgecolor='black')
        axes[1, 1].axvline(x=delay_data.mean(), color='blue', linestyle='--',
                          label=f'Mean: {delay_data.mean():.1f} days')
        axes[1, 1].set_xlabel('Delay Days (for delayed orders)')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Delivery Delay Distribution', fontsize=12, fontweight='bold')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

        plt.suptitle('Vendor Performance Trend Analysis', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()

        return monthly_metrics

    def create_interactive_plotly_dashboard(self):
        """Create interactive dashboard with Plotly"""
        print("\n📊 Creating Interactive Dashboard...")

        # Create figure with subplots
        fig = make_subplots(
            rows=2, cols=3,
            subplot_titles=('Composite Score Distribution',
                           'Performance by Category',
                           'Quality vs Delivery Correlation',
                           'Price Competitiveness Analysis',
                           'Geographic Performance',
                           'Top 10 Vendors'),
            specs=[[{'type': 'histogram'}, {'type': 'bar'}, {'type': 'scatter'}],
                   [{'type': 'scatter'}, {'type': 'bar'}, {'type': 'bar'}]]
        )

        # 1. Composite Score Distribution
        fig.add_trace(
            go.Histogram(x=self.vendor_scores['composite_score'],
                        nbinsx=20, name='Score Distribution',
                        marker_color='steelblue'),
            row=1, col=1
        )

        # 2. Performance by Category
        category_avg = self.vendor_scores.groupby('vendor_category')['composite_score'].mean().sort_values()
        fig.add_trace(
            go.Bar(x=category_avg.index, y=category_avg.values,
                  marker_color='teal', name='Category Avg'),
            row=1, col=2
        )

        # 3. Quality vs Delivery
        fig.add_trace(
            go.Scatter(x=self.vendor_scores['quality_subscore'],
                      y=self.vendor_scores['delivery_subscore'],
                      mode='markers',
                      marker=dict(
                          size=10,
                          color=self.vendor_scores['composite_score'],
                          colorscale='Viridis',
                          showscale=True,
                          colorbar=dict(title='Composite Score')
                      ),
                      text=self.vendor_scores.index,
                      name='Vendors'),
            row=1, col=3
        )

        # 4. Price Competitiveness
        fig.add_trace(
            go.Scatter(x=self.vendor_scores['price_index'],
                      y=self.vendor_scores['composite_score'],
                      mode='markers',
                      marker=dict(
                          size=12,
                          color=self.vendor_scores['tier'].map(
                              {'Tier 1 - Strategic': '#2E7D32',
                               'Tier 2 - Preferred': '#FF9800',
                               'Tier 3 - Transactional': '#D32F2F'}
                          ),
                          symbol=self.vendor_scores['tier'].map(
                              {'Tier 1 - Strategic': 'circle',
                               'Tier 2 - Preferred': 'square',
                               'Tier 3 - Transactional': 'diamond'}
                          )
                      ),
                      text=self.vendor_scores['vendor_name'],
                      name='Price vs Performance'),
            row=2, col=1
        )

        # Add reference line for price index
        fig.add_shape(
            type="line",
            x0=100, y0=0, x1=100, y1=100,
            line=dict(color="Red", width=2, dash="dash"),
            row=2, col=1
        )

        # 5. Geographic Performance
        location_avg = self.vendor_scores.groupby('vendor_location')['composite_score'].mean().sort_values()
        fig.add_trace(
            go.Bar(x=location_avg.index, y=location_avg.values,
                  marker_color='purple', name='Location Avg'),
            row=2, col=2
        )

        # 6. Top 10 Vendors
        top_10 = self.vendor_scores.nlargest(10, 'composite_score')
        fig.add_trace(
            go.Bar(x=top_10.index, y=top_10['composite_score'],
                  marker_color='green', name='Top 10',
                  text=top_10['vendor_name'],
                  textposition='outside'),
            row=2, col=3
        )

        # Update layout
        fig.update_layout(
            height=800,
            showlegend=False,
            title_text="Interactive Vendor Performance Dashboard",
            title_font=dict(size=20, weight='bold')
        )

        fig.show()

        return fig

# Run advanced analytics
print("\n" + "=" * 60)
print("📈 ADVANCED ANALYTICS & VISUALIZATIONS")
print("=" * 60)

analytics = AdvancedAnalytics(vendor_scores, enhanced_df)

# Create comprehensive dashboard
analytics.create_comprehensive_dashboard()

# Create interactive dashboard
analytics.create_interactive_plotly_dashboard()

# Create radar chart for top 3 vendors
top_3_vendors = vendor_scores.nlargest(3, 'composite_score').index.tolist()
analytics.create_radar_chart(top_3_vendors)

# Create trend analysis
trend_data = analytics.create_trend_analysis()

#7. MACHINE LEARNING FOR PREDICTIVE ANALYTICS

class PredictiveAnalytics:
    """Machine learning models for predictive vendor analytics"""

    def __init__(self, vendor_scores, enhanced_df):
        self.vendor_scores = vendor_scores
        self.enhanced_df = enhanced_df
        self.models = {}

    def prepare_ml_data(self):
        """Prepare data for machine learning"""
        print("🔧 Preparing ML Dataset...")

        # Create feature matrix
        features = [
            'quality_subscore', 'delivery_subscore', 'price_subscore',
            'response_subscore', 'invoice_subscore', 'sustainability_subscore',
            'rejection_rate', 'ontime_rate', 'fulfillment_rate',
            'price_index', 'communication_score', 'accuracy_score'
        ]

        # Ensure all features exist
        available_features = [f for f in features if f in self.vendor_scores.columns]

        X = self.vendor_scores[available_features].fillna(0)
        y = self.vendor_scores['composite_score']

        # Create classification target
        y_class = self.vendor_scores['tier'].map({
            'Tier 1 - Strategic': 0,
            'Tier 2 - Preferred': 1,
            'Tier 3 - Transactional': 2
        })

        print(f"✅ ML Data Prepared:")
        print(f"   Features: {X.shape[1]}")
        print(f"   Samples: {X.shape[0]}")
        print(f"   Features used: {available_features}")

        return X, y, y_class, available_features

    def train_regression_models(self):
        """Train regression models to predict composite scores"""
        from sklearn.model_selection import train_test_split, cross_val_score
        from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
        from sklearn.linear_model import LinearRegression, Ridge
        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

        print("\n🤖 Training Regression Models...")

        X, y, _, features = self.prepare_ml_data()

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        # Define models
        models = {
            'Linear Regression': LinearRegression(),
            'Ridge Regression': Ridge(alpha=1.0),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
        }

        results = {}

        for name, model in models.items():
            print(f"  Training {name}...")

            # Train model
            model.fit(X_train, y_train)

            # Make predictions
            y_pred = model.predict(X_test)

            # Calculate metrics
            mae = mean_absolute_error(y_test, y_pred)
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            r2 = r2_score(y_test, y_pred)

            # Cross-validation
            cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
            cv_mean = cv_scores.mean()
            cv_std = cv_scores.std()

            results[name] = {
                'model': model,
                'mae': mae,
                'rmse': rmse,
                'r2': r2,
                'cv_mean': cv_mean,
                'cv_std': cv_std
            }

            print(f"    R²: {r2:.3f}, RMSE: {rmse:.3f}, CV R²: {cv_mean:.3f} ± {cv_std:.3f}")

        # Store best model
        best_model_name = max(results, key=lambda x: results[x]['r2'])
        self.models['regression'] = results[best_model_name]['model']

        # Plot results
        self.plot_regression_results(results)

        return results

    def plot_regression_results(self, results):
        """Visualize regression model performance"""
        from sklearn.model_selection import train_test_split
        import pandas  as pd
        import numpy as np
        import matplotlib.pyplot as plt


        X, y, _, _ = self.prepare_ml_data()
        best_model_name = max(results, key=lambda x: results[x]['r2'])
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        # Model comparison
        model_names = list(results.keys())
        r2_scores = [results[m]['r2'] for m in model_names]
        rmse_scores = [results[m]['rmse'] for m in model_names]

        axes[0, 0].bar(model_names, r2_scores, color='steelblue', alpha=0.7)
        axes[0, 0].set_title('R² Score Comparison', fontweight='bold')
        axes[0, 0].set_ylabel('R² Score')
        axes[0, 0].tick_params(axis='x', rotation=45)
        axes[0, 0].grid(True, alpha=0.3)

        axes[0, 1].bar(model_names, rmse_scores, color='coral', alpha=0.7)
        axes[0, 1].set_title('RMSE Comparison', fontweight='bold')
        axes[0, 1].set_ylabel('RMSE')
        axes[0, 1].tick_params(axis='x', rotation=45)
        axes[0, 1].grid(True, alpha=0.3)

        # Feature importance for Random Forest
        if 'Random Forest' in results:
            rf_model = results['Random Forest']['model']
            feature_importance = pd.DataFrame({
                'feature': X.columns,
                'importance': rf_model.feature_importances_
            }).sort_values('importance', ascending=False)

            axes[1, 0].barh(feature_importance['feature'][:10],
                           feature_importance['importance'][:10],
                           color='teal', alpha=0.7)
            axes[1, 0].set_title('Top 10 Feature Importance (Random Forest)', fontweight='bold')
            axes[1, 0].set_xlabel('Importance')
            axes[1, 0].invert_yaxis()
            axes[1, 0].grid(True, alpha=0.3)

        # Prediction vs Actual
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        best_model = self.models['regression']
        y_pred = best_model.predict(X_test)

        axes[1, 1].scatter(y_test, y_pred, alpha=0.6, color='purple')
        axes[1, 1].plot([y_test.min(), y_test.max()],
                       [y_test.min(), y_test.max()],
                       'r--', lw=2, label='Perfect Prediction')
        axes[1, 1].set_xlabel('Actual Score')
        axes[1, 1].set_ylabel('Predicted Score')
        axes[1, 1].set_title(f'Actual vs Predicted ({best_model_name})', fontweight='bold')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

        plt.suptitle('Regression Model Performance Analysis', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def train_classification_models(self):
        """Train classification models for tier prediction"""
        from sklearn.model_selection import train_test_split
        from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, ConfusionMatrixDisplay

        print("\n🤖 Training Classification Models...")

        X, _, y_class, features = self.prepare_ml_data()

        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y_class, test_size=0.2, random_state=42, stratify=y_class
        )

        # Define models
        models = {
            'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
        }

        results = {}

        for name, model in models.items():
            print(f"  Training {name}...")

            # Train model
            model.fit(X_train, y_train)

            # Make predictions
            y_pred = model.predict(X_test)

            # Calculate metrics
            accuracy = accuracy_score(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)

            results[name] = {
                'model': model,
                'accuracy': accuracy,
                'report': report
            }

            print(f"    Accuracy: {accuracy:.3f}")

        # Store best model
        best_model_name = max(results, key=lambda x: results[x]['accuracy'])
        self.models['classification'] = results[best_model_name]['model']

        # Plot confusion matrix for best model
        self.plot_confusion_matrix(results[best_model_name], X_test, y_test, best_model_name)

        return results

    def plot_confusion_matrix(self, model_result, X_test, y_test, model_name):
        """Plot confusion matrix for classification"""
        from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

        y_pred = model_result['model'].predict(X_test)

        fig, ax = plt.subplots(figsize=(8, 6))
        cm = confusion_matrix(y_test, y_pred)

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=['Tier 1', 'Tier 2', 'Tier 3']
        )

        disp.plot(ax=ax, cmap='Blues', values_format='d')
        ax.set_title(f'Confusion Matrix - {model_name}\nAccuracy: {model_result["accuracy"]:.3f}',
                    fontweight='bold')
        plt.tight_layout()
        plt.show()

    def predict_vendor_tier(self, vendor_features):
        """Predict tier for new vendor based on features"""
        if 'classification' not in self.models:
            print("⚠️ Classification model not trained. Training now...")
            self.train_classification_models()

        model = self.models['classification']

        # Prepare features
        features_df = pd.DataFrame([vendor_features])

        # Make prediction
        prediction = model.predict(features_df)[0]
        probabilities = model.predict_proba(features_df)[0]

        tier_mapping = {
            0: 'Tier 1 - Strategic',
            1: 'Tier 2 - Preferred',
            2: 'Tier 3 - Transactional'
        }

        return {
            'predicted_tier': tier_mapping[prediction],
            'probabilities': {
                tier_mapping[i]: f"{prob:.1%}"
                for i, prob in enumerate(probabilities)
            }
        }

    def cluster_vendors(self, n_clusters=3):
        """Cluster vendors using K-means"""
        from sklearn.cluster import KMeans
        from sklearn.preprocessing import StandardScaler

        print(f"\n🔍 Clustering vendors into {n_clusters} groups...")

        X, _, _, _ = self.prepare_ml_data()

        # Scale features
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)

        # Apply K-means
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        clusters = kmeans.fit_predict(X_scaled)

        # Add clusters to vendor scores
        self.vendor_scores['cluster'] = clusters

        # Analyze clusters
        cluster_analysis = self.vendor_scores.groupby('cluster').agg({
            'composite_score': ['mean', 'std', 'count'],
            'quality_subscore': 'mean',
            'delivery_subscore': 'mean',
            'price_subscore': 'mean'
        }).round(2)

        print("\n📊 Cluster Analysis:")
        display(cluster_analysis)

        # Visualize clusters
        self.plot_clusters(X_scaled, clusters, kmeans.cluster_centers_)

        return clusters

    def plot_clusters(self, X_scaled, clusters, centroids):
        """Visualize vendor clusters"""
        from sklearn.decomposition import PCA

        # Reduce dimensions for visualization
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(X_scaled)

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Scatter plot of clusters
        scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                                 c=clusters, cmap='tab10', s=100, alpha=0.6)
        axes[0].set_xlabel('PCA Component 1')
        axes[0].set_ylabel('PCA Component 2')
        axes[0].set_title('Vendor Clusters (PCA Visualization)', fontweight='bold')
        axes[0].grid(True, alpha=0.3)

        # Add centroids
        if centroids is not None:
            centroids_pca = pca.transform(centroids)
            axes[0].scatter(centroids_pca[:, 0], centroids_pca[:, 1],
                           marker='X', s=200, c='red', label='Centroids')
            axes[0].legend()

        # Cluster distribution
        cluster_counts = pd.Series(clusters).value_counts().sort_index()
        axes[1].bar(cluster_counts.index.astype(str), cluster_counts.values,
                   color=plt.cm.tab10(range(len(cluster_counts))), alpha=0.7)
        axes[1].set_xlabel('Cluster')
        axes[1].set_ylabel('Number of Vendors')
        axes[1].set_title('Vendor Distribution Across Clusters', fontweight='bold')
        axes[1].grid(True, alpha=0.3)

        plt.suptitle('Vendor Segmentation Analysis', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

    def create_recommendation_engine(self):
        """Create vendor recommendation engine"""
        print("\n💡 Building Recommendation Engine...")

        # Calculate similarity matrix
        X, _, _, _ = self.prepare_ml_data()

        from sklearn.metrics.pairwise import cosine_similarity

        # Calculate similarity between vendors
        similarity_matrix = cosine_similarity(X)
        similarity_df = pd.DataFrame(
            similarity_matrix,
            index=self.vendor_scores.index,
            columns=self.vendor_scores.index
        )

        # Function to get recommendations
        def get_similar_vendors(vendor_code, n_recommendations=5):
            """Get similar vendors based on performance profile"""
            if vendor_code not in similarity_df.index:
                return None

            similar = similarity_df[vendor_code].sort_values(ascending=False)
            similar = similar.drop(vendor_code)  # Remove self

            recommendations = []
            for idx, score in similar.head(n_recommendations).items():
                vendor_data = self.vendor_scores.loc[idx]
                recommendations.append({
                    'supplier_code': idx,
                    'vendor_name': vendor_data['vendor_name'],
                    'similarity_score': round(score, 3),
                    'composite_score': vendor_data['composite_score'],
                    'tier': vendor_data['tier'],
                    'category': vendor_data['vendor_category']
                })

            return pd.DataFrame(recommendations)

        self.recommendation_engine = get_similar_vendors

        print("✅ Recommendation engine ready!")

        # Test with sample vendor
        sample_vendor = self.vendor_scores.index[0]
        recommendations = get_similar_vendors(sample_vendor, 3)

        if recommendations is not None:
            print(f"\n📋 Similar vendors to {sample_vendor}:")
            display(recommendations)

        return get_similar_vendors

# Run predictive analytics
print("\n" + "=" * 60)
print("🤖 PREDICTIVE ANALYTICS & ML MODELS")
print("=" * 60)

# First check if vendor_scores exists, if not create it
if 'vendor_scores' not in locals():
    print("⚠️ vendor_scores not found. Running scoring engine first...")
    scoring_engine = VendorScoringEngine(enhanced_df)
    vendor_scores = scoring_engine.calculate_composite_scores()

predictive = PredictiveAnalytics(vendor_scores, enhanced_df)
# Prepare ML data
X, y, y_class, features = predictive.prepare_ml_data()

# Train regression models
regression_results = predictive.train_regression_models()

# Train classification models
classification_results = predictive.train_classification_models()

# Cluster vendors
clusters = predictive.cluster_vendors(n_clusters=3)

# Create recommendation engine
recommendation_engine = predictive.create_recommendation_engine()

# Test prediction with sample vendor
sample_features = {
    'quality_subscore': 85,
    'delivery_subscore': 90,
    'price_subscore': 80,
    'response_subscore': 75,
    'invoice_subscore': 88,
    'sustainability_subscore': 70,
    'rejection_rate': 2.5,
    'ontime_rate': 0.92,
    'fulfillment_rate': 95.0,
    'price_index': 105.0,
    'communication_score': 85.0,
    'accuracy_score': 90.0
}

# Only include features that exist in our model
available_features = {k: v for k, v in sample_features.items() if k in features}
prediction = predictive.predict_vendor_tier(available_features)

print(f"\n🔮 Sample Tier Prediction:")
print(f"   Predicted Tier: {prediction['predicted_tier']}")
print(f"   Tier Probabilities:")
for tier, prob in prediction['probabilities'].items():
    print(f"     {tier}: {prob}")

#8. REPORT GENERATION & EXPORT

class ReportGenerator:
    """Generate comprehensive reports and export results"""

    def __init__(self, vendor_scores, enhanced_df, scoring_engine, predictive_analytics):
        self.vendor_scores = vendor_scores
        self.enhanced_df = enhanced_df
        self.scoring_engine = scoring_engine
        self.predictive = predictive_analytics
        self.timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    def generate_executive_summary(self):
        """Generate executive summary report"""
        print("\n" + "=" * 60)
        print("📋 GENERATING EXECUTIVE SUMMARY")
        print("=" * 60)

        summary = {
            'report_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'total_vendors': len(self.vendor_scores),
            'total_purchase_orders': len(self.enhanced_df),
            'analysis_period': f"{self.enhanced_df['gr_date'].min()} to {self.enhanced_df['gr_date'].max()}",
            'tier_distribution': self.vendor_scores['tier'].value_counts().to_dict(),
            'avg_composite_score': round(self.vendor_scores['composite_score'].mean(), 2),
            'top_performer': {
                'code': self.vendor_scores.nlargest(1, 'composite_score').index[0],
                'name': self.vendor_scores.nlargest(1, 'composite_score')['vendor_name'].iloc[0],
                'score': round(self.vendor_scores['composite_score'].max(), 1)
            },
            'category_performance': self.vendor_scores.groupby('vendor_category')['composite_score']
                                    .mean().round(2).to_dict(),
            'key_insights': self._generate_insights()
        }

        return summary

    def _generate_insights(self):
        """Generate key insights from analysis"""
        insights = []

        # Tier distribution insight
        tier_pct = self.vendor_scores['tier'].value_counts(normalize=True) * 100
        if tier_pct.get('Tier 1 - Strategic', 0) > 30:
            insights.append("Strong vendor base with over 30% in Tier 1")
        elif tier_pct.get('Tier 1 - Strategic', 0) < 15:
            insights.append("Opportunity to develop more strategic vendors (less than 15% in Tier 1)")

        # Price vs quality correlation
        price_quality_corr = self.vendor_scores['price_subscore'].corr(self.vendor_scores['quality_subscore'])
        if price_quality_corr < -0.3:
            insights.append("Higher quality often comes at higher cost (negative price-quality correlation)")
        elif price_quality_corr > 0.3:
            insights.append("Vendors achieving both quality and price competitiveness identified")

        # Delivery performance insight
        avg_delivery_score = self.vendor_scores['delivery_subscore'].mean()
        if avg_delivery_score > 85:
            insights.append("Excellent overall delivery performance across vendor base")
        elif avg_delivery_score < 70:
            insights.append("Delivery performance needs improvement across vendor base")

        # Risk concentration
        high_risk_vendors = len(self.vendor_scores[self.vendor_scores['risk_score'] > 70])
        if high_risk_vendors > len(self.vendor_scores) * 0.2:
            insights.append(f"High risk concentration: {high_risk_vendors} vendors with risk score > 70")

        # Sustainability performance
        avg_sustainability = self.vendor_scores['sustainability_subscore'].mean()
        if avg_sustainability < 60:
            insights.append("Sustainability performance needs improvement across vendor base")

        return insights

    def export_to_excel(self):
        """Export comprehensive analysis to Excel"""
        print("\n💾 Exporting to Excel...")

        filename = f'vendor_analysis_report_{self.timestamp}.xlsx'

        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            # 1. Vendor Scores
            self.vendor_scores.to_excel(writer, sheet_name='Vendor Scores')

            # 2. Executive Summary
            summary = self.generate_executive_summary()
            summary_df = pd.DataFrame([summary])
            summary_df.to_excel(writer, sheet_name='Executive Summary')

            # 3. Detailed Metrics
            detailed_metrics = self.vendor_scores.copy()
            detailed_metrics.to_excel(writer, sheet_name='Detailed Metrics')

            # 4. Tier Analysis
            tier_analysis = self.vendor_scores.groupby('tier').agg({
                'composite_score': ['count', 'mean', 'std', 'min', 'max'],
                'vendor_name': lambda x: ', '.join(x)
            }).round(2)
            tier_analysis.to_excel(writer, sheet_name='Tier Analysis')

            # 5. Category Analysis
            category_analysis = self.vendor_scores.groupby('vendor_category').agg({
                'composite_score': ['mean', 'std', 'count'],
                'quality_subscore': 'mean',
                'delivery_subscore': 'mean',
                'price_subscore': 'mean'
            }).round(2)
            category_analysis.to_excel(writer, sheet_name='Category Analysis')

            # 6. Improvement Candidates
            improvement_candidates = self.vendor_scores[
                self.vendor_scores['tier'] == 'Tier 3 - Transactional'
            ].sort_values('composite_score').head(10)
            improvement_candidates.to_excel(writer, sheet_name='Improvement Candidates')

            # 7. Top Performers
            top_performers = self.vendor_scores[
                self.vendor_scores['tier'] == 'Tier 1 - Strategic'
            ].sort_values('composite_score', ascending=False).head(10)
            top_performers.to_excel(writer, sheet_name='Top Performers')

            # 8. Raw Data Sample
            self.enhanced_df.head(100).to_excel(writer, sheet_name='Raw Data Sample')

        print(f"✅ Excel report saved as: {filename}")

        # Download the file
        files.download(filename)

        return filename

    def generate_text_report(self):
        """Generate detailed text report"""
        print("\n📄 Generating Detailed Report...")

        summary = self.generate_executive_summary()

        report_lines = []
        report_lines.append("=" * 80)
        report_lines.append("VENDOR PERFORMANCE ANALYTICS REPORT")
        report_lines.append("=" * 80)
        report_lines.append(f"\nReport Generated: {summary['report_date']}")
        report_lines.append(f"Analysis Period: {summary['analysis_period']}")

        report_lines.append("\n" + "-" * 80)
        report_lines.append("EXECUTIVE SUMMARY")
        report_lines.append("-" * 80)
        report_lines.append(f"\nTotal Vendors Analyzed: {summary['total_vendors']}")
        report_lines.append(f"Total Purchase Orders: {summary['total_purchase_orders']}")
        report_lines.append(f"Average Composite Score: {summary['avg_composite_score']}")

        report_lines.append("\nTier Distribution:")
        for tier, count in summary['tier_distribution'].items():
            pct = (count / summary['total_vendors']) * 100
            report_lines.append(f"  {tier}: {count} vendors ({pct:.1f}%)")

        report_lines.append(f"\nTop Performer:")
        report_lines.append(f"  {summary['top_performer']['name']} ({summary['top_performer']['code']})")
        report_lines.append(f"  Score: {summary['top_performer']['score']}")

        report_lines.append("\nCategory Performance (Average Scores):")
        for category, score in summary['category_performance'].items():
            report_lines.append(f"  {category}: {score}")

        report_lines.append("\n" + "-" * 80)
        report_lines.append("KEY INSIGHTS")
        report_lines.append("-" * 80)
        for i, insight in enumerate(summary['key_insights'], 1):
            report_lines.append(f"{i}. {insight}")

        report_lines.append("\n" + "-" * 80)
        report_lines.append("STRATEGIC RECOMMENDATIONS")
        report_lines.append("-" * 80)

        recommendations = [
            "Increase procurement share with Tier 1 vendors to 60% of total spend",
            "Establish vendor development programs for Tier 2 vendors with potential",
            "Develop exit strategy for bottom 20% of Tier 3 vendors within 6 months",
            "Implement quarterly business reviews with all Tier 1 vendors",
            "Create category-specific improvement initiatives based on analysis",
            "Establish clear KPIs and scorecards for all strategic vendors",
            "Consider vendor consolidation in underperforming categories",
            "Implement automated vendor performance monitoring system",
            "Develop sustainability improvement programs for low-scoring vendors",
            "Establish price benchmarking against category medians quarterly"
        ]

        for i, rec in enumerate(recommendations, 1):
            report_lines.append(f"{i}. {rec}")

        report_lines.append("\n" + "-" * 80)
        report_lines.append("NEXT STEPS")
        report_lines.append("-" * 80)

        next_steps = [
            "Review detailed vendor scores in attached Excel report",
            "Schedule meetings with top 10 vendors to strengthen partnerships",
            "Develop action plans for bottom 10 vendors",
            "Implement tracking for key improvement metrics",
            "Schedule quarterly review of this analysis"
        ]

        for i, step in enumerate(next_steps, 1):
            report_lines.append(f"{i}. {step}")

        report_lines.append("\n" + "=" * 80)
        report_lines.append("END OF REPORT")
        report_lines.append("=" * 80)

        report_text = "\n".join(report_lines)

        # Save to file
        text_filename = f'vendor_report_{self.timestamp}.txt'
        with open(text_filename, 'w') as f:
            f.write(report_text)

        print(f"✅ Text report saved as: {text_filename}")

        # Download the file
        files.download(text_filename)

        return report_text

    def create_presentation_slides(self):
        """Create PowerPoint-like summary slides"""
        print("\n📊 Creating Presentation Summary...")

        slides = []

        # Slide 1: Title Slide
        slides.append({
            'title': 'Vendor Performance Analytics',
            'subtitle': f'Analysis Date: {datetime.now().strftime("%Y-%m-%d")}',
            'content': [
                f'Total Vendors: {len(self.vendor_scores)}',
                f'Total Purchase Orders: {len(self.enhanced_df)}',
                f'Analysis Period: {self.enhanced_df["gr_date"].min()} to {self.enhanced_df["gr_date"].max()}'
            ]
        })

        # Slide 2: Tier Distribution
        tier_dist = self.vendor_scores['tier'].value_counts()
        slides.append({
            'title': 'Vendor Tier Distribution',
            'content': [f'{tier}: {count} vendors ({(count/len(self.vendor_scores)*100):.1f}%)'
                       for tier, count in tier_dist.items()]
        })

        # Slide 3: Top Performers
        top_5 = self.vendor_scores.nlargest(5, 'composite_score')
        slides.append({
            'title': 'Top 5 Performing Vendors',
            'content': [f'{row["vendor_name"]} ({idx}): Score = {row["composite_score"]}, Tier = {row["tier"]}'
                       for idx, row in top_5.iterrows()]
        })

        # Slide 4: Category Performance
        category_avg = self.vendor_scores.groupby('vendor_category')['composite_score'].mean().round(2)
        slides.append({
            'title': 'Performance by Category',
            'content': [f'{category}: {score}' for category, score in category_avg.items()]
        })

        # Slide 5: Key Insights
        insights = self._generate_insights()
        slides.append({
            'title': 'Key Insights',
            'content': insights[:5]  # Top 5 insights
        })

        # Slide 6: Recommendations
        slides.append({
            'title': 'Strategic Recommendations',
            'content': [
                'Increase spend with Tier 1 vendors',
                'Develop improvement programs for Tier 2',
                'Exit strategy for bottom Tier 3 vendors',
                'Implement quarterly performance reviews',
                'Establish category-specific initiatives'
            ]
        })

        # Display slides
        print("\n" + "=" * 60)
        print("📊 PRESENTATION SUMMARY")
        print("=" * 60)

        for i, slide in enumerate(slides, 1):
            print(f"\nSlide {i}: {slide['title']}")
            print("-" * 40)
            for item in slide['content']:
                print(f"  • {item}")

        return slides

# Generate reports
print("\n" + "=" * 60)
print("📋 REPORT GENERATION & EXPORT")
print("=" * 60)

report_generator = ReportGenerator(vendor_scores, enhanced_df, scoring_engine, predictive)

# Generate executive summary
executive_summary = report_generator.generate_executive_summary()
print("\n📊 Executive Summary:")
for key, value in executive_summary.items():
    if key != 'key_insights':
        print(f"  {key.replace('_', ' ').title()}: {value}")

# Export to Excel
excel_file = report_generator.export_to_excel()

# Generate text report
text_report = report_generator.generate_text_report()

# Create presentation slides
slides = report_generator.create_presentation_slides()


#10. FINAL SUMMARY & NEXT STEPS
print("\n" + "=" * 80)
print("🎯 VENDOR ANALYTICS PIPELINE - COMPLETE SUMMARY")
print("=" * 80)

print("\n✅ ANALYSIS COMPLETED SUCCESSFULLY!")

print("\n📊 KEY ACHIEVEMENTS:")
print(f"   • Vendors Analyzed: {len(vendor_scores)}")
print(f"   • Purchase Orders Processed: {len(enhanced_df)}")
print(f"   • Performance Dimensions: 6")
print(f"   • ML Models Trained: 7")
print(f"   • Reports Generated: 3")

print("\n📈 VENDOR TIER DISTRIBUTION:")
tier_dist = vendor_scores['tier'].value_counts()
for tier, count in tier_dist.items():
    pct = (count / len(vendor_scores)) * 100
    print(f"   • {tier}: {count} vendors ({pct:.1f}%)")

print("\n🏆 TOP 3 VENDORS:")
top_3 = vendor_scores.nlargest(3, 'composite_score')
for idx, (vendor_code, row) in enumerate(top_3.iterrows(), 1):
    print(f"   {idx}. {row['vendor_name']} ({vendor_code})")
    print(f"      Score: {row['composite_score']}, Tier: {row['tier']}")
    print(f"      Category: {row['vendor_category']}, Location: {row['vendor_location']}")

print("\n⚠️ BOTTOM 3 VENDORS (Need Attention):")
bottom_3 = vendor_scores.nsmallest(3, 'composite_score')
for idx, (vendor_code, row) in enumerate(bottom_3.iterrows(), 1):
    print(f"   {idx}. {row['vendor_name']} ({vendor_code})")
    print(f"      Score: {row['composite_score']}, Tier: {row['tier']}")
    print(f"      Risk Score: {row['risk_score']}")

print("\n📁 FILES GENERATED:")
print("   1. Excel Report: vendor_analysis_report_YYYYMMDD_HHMMSS.xlsx")
print("   2. Text Report: vendor_report_YYYYMMDD_HHMMSS.txt")
print("   3. Visualizations: Displayed inline in notebook")

print("\n🔧 ML MODELS READY FOR USE:")
print("   1. Regression Models: Predict vendor composite scores")
print("   2. Classification Models: Predict vendor tiers")
print("   3. Clustering Models: Segment vendors into groups")
print("   4. Recommendation Engine: Find similar vendors")

print("\n🚀 IMMEDIATE NEXT STEPS:")
print("   1. Review the Excel report for detailed analysis")
print("   2. Schedule meetings with top performing vendors")
print("   3. Develop improvement plans for bottom performers")
print("   4. Integrate results into your procurement system")
print("   5. Schedule monthly re-runs for continuous monitoring")

print("\n" + "=" * 80)
print("🎉 ANALYSIS COMPLETE! READY FOR PRODUCTION DEPLOYMENT")
print("=" * 80)

# Save final state for reference
print("\n💾 Saving analysis state...")
analysis_state = {
    'vendor_scores': vendor_scores,
    'enhanced_df': enhanced_df,
    'scoring_engine': scoring_engine,
    'predictive_analytics': predictive,
    'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S')
}

print("✅ Analysis state saved. You can now:")
print("   - Download the reports")
print("   - Integrate with your systems")
print("   - Schedule regular analysis")
print("   - Extend with custom business rules")

✅ Libraries imported successfully!
📅 Analysis Date: 2026-10-22 06:20:02
📊 DATA LOADING MODULE
📤 Please upload your Vendor Master CSV file:


⚠️ No vendor CSV uploaded. Using sample data...

📤 Please upload your Purchase Orders CSV file:
